# 06 — Places theme

**Feature type:** `place`
**Geometry:** point.

A place represents a real-world entity such as a business, institution,
landmark, or geographic feature. Categories, confidence, brand, contact
arrays, addresses, names, status, and provenance are distinct concepts; ETL
should not flatten them into one untyped label.

In [ ]:
from pyspark.sql import functions as F
from overture_lab.config import load_settings
from overture_lab.spark import create_sedona
from overture_lab.regions import resolve_focus_regions
from overture_lab.lesson import inspect_type

settings = load_settings()
spark = create_sedona(settings, "06-places")
regions = resolve_focus_regions(spark, settings)
places = inspect_type(
    spark,
    settings,
    regions,
    "places",
    "place",
    ["id", "names", "categories", "basic_category", "taxonomy", "confidence", "brand", "addresses", "websites", "phones", "operating_status", "sources", "geometry"],
)
display(places["schema"])

## Category model

`categories.primary` is the detailed primary category;
`categories.alternate` carries alternatives; `basic_category` is a broader
simplified category; `taxonomy` retains a hierarchy.

In [ ]:
display(
    places["country"]
    .groupBy("basic_category", "categories.primary")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
    .toPandas()
)

## Inspect nested records without losing arrays

In [ ]:
display(
    places["locality"].select(
        "id",
        F.col("names.primary").alias("primary_name"),
        F.element_at("names.common", F.lit("en")).alias("english_name"),
        F.col("categories.primary").alias("primary_category"),
        "basic_category",
        "confidence",
        F.col("brand.names.primary").alias("brand_name"),
        F.element_at("addresses", 1).alias("first_address"),
        F.size("websites").alias("website_count"),
        F.size("phones").alias("phone_count"),
        "operating_status",
    ).limit(25).toPandas()
)

## Confidence distribution

Null confidence means “no confidence information,” not zero confidence.

In [ ]:
display(
    places["country"].select(
        F.when(F.col("confidence").isNull(), "missing")
        .when(F.col("confidence") < 0.5, "below_0.5")
        .when(F.col("confidence") < 0.8, "0.5_to_0.8")
        .otherwise("0.8_to_1.0")
        .alias("confidence_band")
    ).groupBy("confidence_band").count().orderBy("confidence_band").toPandas()
)

## Place map by broad category

In [ ]:
from overture_lab.visualize import static_geometry_plot, interactive_geometry_map

mapped, axis = static_geometry_plot(
    places["locality"],
    limit=settings.map_feature_limit,
    columns=["basic_category", "confidence", "geometry"],
    column="basic_category",
    title=f"Places in {settings.locality_name_en} by broad category",
)

# The interactive deck has no network basemap. VS Code renders the layer from
# the bounded local data only.
interactive_geometry_map(
    places["locality"],
    limit=min(settings.map_feature_limit, 500),
    columns=["basic_category", "confidence", "geometry"],
)